# Town-Level Dynasty Stronghold % & Curated Partitioning


In [1]:
import sys
import numpy as np
import pandas as pd
sys.path.insert(0, "..")

S = "../data/staging/"
C = "../data/curated/"
people = pd.read_parquet(S + "politicians.parquet")
terms = pd.read_parquet(S + "politician_terms.parquet")
edges = pd.read_parquet(S + "kinship_edges.parquet")
clans = pd.read_parquet(S + "clans.parquet")
memberships = pd.read_parquet(S + "hf_memberships_clean.parquet")   # used to know which towns are cities

ELECTION_YEARS = [1988, 1992, 1995, 1998, 2001, 2004, 2007, 2010, 2013, 2016, 2019, 2022, 2025]
LOCAL = ["MAYOR", "VICE MAYOR", "COUNCILOR"]
PROVINCIAL = ["GOVERNOR", "VICE GOVERNOR", "PROVINCIAL BOARD MEMBER"]
print(f"people: {len(people):,} | terms: {len(terms):,} | kinship links: {len(edges):,} | clans: {len(clans):,}")

people: 115,954 | terms: 316,169 | kinship links: 288,180 | clans: 20,394


The rule: an official is dynastic in a given year if at least one strong or medium relative held office in that year or earlier. A relative who only enters politics later doesn't count yet. So a mayor elected in 2004 whose son first wins in 2019 becomes "dynastic" only from 2019 onward, which is fair.

We also mark a stricter version, the **fat dynasty** (the definition used by the Ateneo Policy Center): a relative holds office in the **same province in the same year**. This lets us compare our numbers with Ateneo's published figures later.

Our data starts in 1988, so relatives who served before that are not visible. That is why the 1988 share is low and rises over the next elections.

# Mark each official as dynastic, per election

In [2]:
info = people.set_index("person_uid")

# Strong/medium links, without same-name "twins" (probably one person under two IDs)
sm = edges[edges["tier"].isin(["strong", "medium"])]
A, B = info.loc[sm["person_a"]], info.loc[sm["person_b"]]
same_name = (A["last_key"].to_numpy() == B["last_key"].to_numpy()) & (A["first_key"].to_numpy() == B["first_key"].to_numpy())
mid_ok = A["middle_key"].isna().to_numpy() | B["middle_key"].isna().to_numpy() | (A["middle_key"].to_numpy() == B["middle_key"].to_numpy())
twin = same_name & mid_ok
sm = sm[~twin]
print(f"links skipped as possible same-person twins: {twin.sum():,}")

# For each person: the earliest year any of their relatives held office (primary terms only)
first_primary_year = terms[terms["is_primary"]].groupby("person_uid")["year"].min()
pairs = pd.concat([sm[["person_a", "person_b"]].set_axis(["person_uid", "relative"], axis=1),
                   sm[["person_b", "person_a"]].set_axis(["person_uid", "relative"], axis=1)])
pairs["relative_first_year"] = pairs["relative"].map(first_primary_year)
first_relative_year = pairs.groupby("person_uid")["relative_first_year"].min()

# One row per official per election: dynastic if a relative was in office that year or earlier
t = terms[terms["is_primary"] & terms["year"].isin(ELECTION_YEARS)].copy()
t["first_relative_year"] = t["person_uid"].map(first_relative_year)
t["is_dynastic"] = t["first_relative_year"].le(t["year"]).fillna(False).astype(bool)
t["clan_id"] = t["person_uid"].map(info["clan_id"])


# "Fat dynasty" (Ateneo's definition): a relative holds office in the SAME province in the SAME year
held = t.dropna(subset=["province_std"])[["person_uid", "year", "province_std"]].drop_duplicates()
together = (pairs.merge(held, on="person_uid")
                 .merge(held.rename(columns={"person_uid": "relative"}), on=["relative", "year", "province_std"]))
fat_keys = set(zip(together["person_uid"], together["year"], together["province_std"]))
t["is_fat_dynasty"] = [k in fat_keys for k in zip(t["person_uid"], t["year"], t["province_std"])]

t.groupby("year").agg(officials=("person_uid", "size"), source=("source", lambda s: s.mode().iloc[0]),
                      dynastic_share=("is_dynastic", "mean"), fat_dynasty_share=("is_fat_dynasty", "mean")).round(3)

links skipped as possible same-person twins: 4,678


,officials,source,dynastic_share,fat_dynasty_share
year,,,,
1988,16623,ateneo,0.081,0.069
1992,17029,ateneo,0.181,0.103
1995,17273,ateneo,0.241,0.123
1998,17221,ateneo,0.299,0.130
2001,17490,openhalalan,0.355,0.148
2004,17372,hf,0.412,0.177
2007,16610,hf,0.464,0.199
2010,17507,hf,0.508,0.233
2013,17164,hf,0.526,0.235


## Town-level stronghold %

In [3]:
# Which towns are cities (HF writes them as "CITY OF ...")
city = memberships.loc[memberships["locality"].fillna("").str.upper().str.contains("CITY"), ["province_std", "locality_std"]]
city = set(map(tuple, city.dropna().drop_duplicates().to_numpy()))

# Local officials with a known town
loc = t[t["position"].isin(LOCAL)].dropna(subset=["town_std"])
keys = ["year", "province_std", "town_std"]

# Stronghold % per town and election
loc_people = loc.drop_duplicates(keys + ["person_uid"])   # count each person once per town (some IDs hold 2 posts)
town = loc_people.groupby(keys).agg(n_officials=("person_uid", "nunique"),
                                    n_dynastic=("is_dynastic", "sum")).reset_index()
town["dynastic_share"] = (town["n_dynastic"] / town["n_officials"]).round(3)

# Is the mayor dynastic?
mayor = loc[loc["position"].eq("MAYOR")].groupby(keys).agg(mayor_uid=("person_uid", "first"),
                                                          mayor_dynastic=("is_dynastic", "max")).reset_index()
town = town.merge(mayor, on=keys, how="left")
town["mayor_dynastic"] = town["mayor_dynastic"].astype("boolean")    # empty when no mayor is recorded

# The family (clan) holding the most seats in each town
top = (loc.dropna(subset=["clan_id"]).groupby(keys + ["clan_id"])["person_uid"].nunique().rename("top_clan_seats")
          .reset_index().sort_values(["top_clan_seats", "clan_id"], ascending=[False, True]).drop_duplicates(keys)
          .rename(columns={"clan_id": "top_clan_id"}))
town = town.merge(top, on=keys, how="left")
town["top_clan_seats"] = town["top_clan_seats"].fillna(0).astype(int)
town["top_clan_share"] = (town["top_clan_seats"] / town["n_officials"]).round(3)
town["top_clan_label"] = town["top_clan_id"].map(clans.set_index("clan_id")["clan_label"])
town["fat_dynasty"] = town["top_clan_seats"] >= 2

town["is_city"] = [(p, c) in city for p, c in zip(town["province_std"], town["town_std"])]
town["stronghold_level"] = pd.cut(town["dynastic_share"], [-0.01, 0.25, 0.5, 0.75, 1.0],
                                  labels=["low (0-25%)", "moderate (25-50%)", "high (50-75%)", "stronghold (75-100%)"])
town["source"] = np.select([town["year"].lt(2001), town["year"].between(2004, 2016)], ["ateneo", "hf"], default="openhalalan")

print(f"town-year rows: {len(town):,}")
display(town.groupby("year").agg(
    towns=("town_std", "size"),
    avg_dynastic_share=("dynastic_share", "mean"),
    dynastic_mayor=("mayor_dynastic", "mean"),
    fat_dynasty=("fat_dynasty", "mean"),
    strongholds=("stronghold_level", lambda s: (s == "stronghold (75-100%)").mean())).round(3))

# Towns where ONE family holds the biggest share of seats (2025)
town[town["year"].eq(2025)].sort_values(["top_clan_share", "n_officials"], ascending=False)[
    ["province_std", "town_std", "n_officials", "dynastic_share", "top_clan_label", "top_clan_seats", "top_clan_share"]].head(10)

town-year rows: 20,764


,towns,avg_dynastic_share,dynastic_mayor,fat_dynasty,strongholds
year,,,,,
1988,1568,0.075,0.166,0.186,0.000
1992,1577,0.174,0.301,0.277,0.001
1995,1598,0.234,0.394,0.317,0.004
1998,1585,0.290,0.478,0.296,0.012
2001,1605,0.345,0.56,0.310,0.022
2004,1591,0.402,0.626,0.345,0.030
2007,1513,0.456,0.67,0.362,0.059
2010,1627,0.503,0.719,0.448,0.114
2013,1592,0.526,0.734,0.483,0.138


,province_std,town_std,n_officials,dynastic_share,top_clan_label,top_clan_seats,top_clan_share
20041,MAGUINDANAO DEL SUR,AMPATUAN,10,0.700,SANGKI | MAGUINDANAO DEL SUR,7,0.700
20059,MAGUINDANAO DEL SUR,SHARIFF AGUAK,10,0.900,AMPATUAN | MAGUINDANAO DEL SUR,7,0.700
20063,MAGUINDANAO DEL SUR,TALAYAN,10,0.900,MIDTIMBANG | MAGUINDANAO DEL SUR,7,0.700
19974,LANAO DEL SUR,PICONG,10,0.700,BALINDONG | LANAO DEL SUR,6,0.600
19979,LANAO DEL SUR,TAGOLOAN II,10,0.800,CAPAL | LANAO DEL SUR,6,0.600
20042,MAGUINDANAO DEL SUR,BULUAN,10,0.700,MANGUDADATU | MAGUINDANAO DEL SUR,6,0.600
20045,MAGUINDANAO DEL SUR,DATU HOFFER AMPATUAN,10,0.700,AMPATUAN | MAGUINDANAO DEL SUR,6,0.600
20052,MAGUINDANAO DEL SUR,MAMASAPANO,9,0.778,AMPATUAN | MAGUINDANAO DEL SUR,5,0.556
19188,AKLAN,LIBACAO,10,0.800,NAVAROSA | AKLAN,5,0.500
19436,CAGAYAN,BUGUEY,10,1.000,ANTIPORDA | CAGAYAN,5,0.500


## Province-level stronghold %

In [4]:
pv = t[t["position"].isin(PROVINCIAL)].dropna(subset=["province_std"])
pk = ["year", "province_std"]

# Provincial officials (governor, vice governor, board members)
pv_people = pv.drop_duplicates(pk + ["person_uid"])   # count each person once per province
prov_tbl = pv_people.groupby(pk).agg(n_provincial_officials=("person_uid", "nunique"),
                                     n_dynastic=("is_dynastic", "sum")).reset_index()
prov_tbl["provincial_dynastic_share"] = (prov_tbl["n_dynastic"] / prov_tbl["n_provincial_officials"]).round(3)

# Is the governor dynastic?
gov = pv[pv["position"].eq("GOVERNOR")].groupby(pk).agg(governor_uid=("person_uid", "first"),
                                                       governor_dynastic=("is_dynastic", "max")).reset_index()

# Summary of the province's towns (from Step 3)
towns_sum = town.groupby(pk).agg(n_towns=("town_std", "size"),
                                 share_towns_dynastic_mayor=("mayor_dynastic", "mean"),
                                 avg_town_dynastic_share=("dynastic_share", "mean"),
                                 n_stronghold_towns=("stronghold_level", lambda s: (s == "stronghold (75-100%)").sum())).reset_index()

# Fat dynasty share over ALL officials in the province (local + provincial + House), Ateneo's definition
fat = t.dropna(subset=["province_std"]).groupby(pk)["is_fat_dynasty"].mean().rename("fat_dynasty_share").round(3).reset_index()
prov_tbl = prov_tbl.merge(gov, on=pk, how="left").merge(towns_sum, on=pk, how="left").merge(fat, on=pk, how="left")
prov_tbl["governor_dynastic"] = prov_tbl["governor_dynastic"].astype("boolean")
cols = ["share_towns_dynastic_mayor", "avg_town_dynastic_share"]
prov_tbl[cols] = prov_tbl[cols].astype(float).round(3)

print(f"province-year rows: {len(prov_tbl):,}")
# The 10 most dynastic provinces in 2025 (by their towns)
prov_tbl[prov_tbl["year"].eq(2025)].sort_values("avg_town_dynastic_share", ascending=False)[
    ["province_std", "governor_dynastic", "provincial_dynastic_share", "n_towns", "avg_town_dynastic_share", "n_stronghold_towns", "fat_dynasty_share"]].head(10)

province-year rows: 1,021


,province_std,governor_dynastic,provincial_dynastic_share,n_towns,avg_town_dynastic_share,n_stronghold_towns,fat_dynasty_share
952,BILIRAN,True,0.800,8.0,0.850,6.0,0.330
949,BATANES,True,0.625,6.0,0.833,5.0,0.290
993,NORTHERN SAMAR,True,0.857,24.0,0.817,20.0,0.320
987,MASBATE,True,0.857,21.0,0.805,15.0,0.370
939,ABRA,True,1.000,27.0,0.789,15.0,0.352
963,CEBU,True,1.000,53.0,0.758,28.0,0.339
960,CAPIZ,True,0.667,17.0,0.753,8.0,0.228
971,EASTERN SAMAR,True,1.000,23.0,0.748,13.0,0.338
976,ILOILO,True,0.857,44.0,0.739,24.0,0.247
957,CAMARINES NORTE,True,0.900,12.0,0.739,5.0,0.265


In our README lists the PCIJ benchmark (113 of 149 cities and 71 of 82 provinces are dynastic) as a validation source. If our numbers land near PCIJ's, our method is believable. This is also one of the data quality checks for Task 10.

In [5]:
bench = pd.DataFrame({
    "cities_with_dynastic_mayor": town[town["is_city"]].groupby("year")["mayor_dynastic"].mean().astype(float),
    "provinces_with_dynastic_governor": prov_tbl.groupby("year")["governor_dynastic"].mean().astype(float)})
pcij = pd.DataFrame({"cities_with_dynastic_mayor": [113 / 149],
                     "provinces_with_dynastic_governor": [71 / 82]}, index=["PCIJ benchmark"])
(pd.concat([bench, pcij]) * 100).round(1).astype(str).add("%")

,cities_with_dynastic_mayor,provinces_with_dynastic_governor
1988,18.5%,42.0%
1992,32.4%,45.2%
1995,36.2%,47.4%
1998,47.8%,52.6%
2001,62.1%,64.1%
2004,66.2%,62.0%
2007,72.0%,63.7%
2010,73.6%,75.6%
2013,76.2%,85.0%
2016,76.4%,81.0%


## Ateneo Policy Center fat dynasty share per province

Ateneo publishes the share of each province's officials who are a fat dynasty (1992-2022). We compare it with our `fat_dynasty_share`. Ateneo uses same surname only, so its shares should be higher than ours (see Task 7, Step 12), but the provinces should rank in a similar order.


In [6]:
# Ateneo's fat dynasty share per province and year (cleaned in Task 6)
ap = pd.read_parquet("../data/staging/ateneo_province_shares_clean.parquet")
ap["ateneo_fat_share"] = ap["fat_dynasty_share_pct"] / 100
cmp = prov_tbl[["year", "province_std", "fat_dynasty_share"]].merge(ap[["year", "province_std", "ateneo_fat_share"]], on=pk)
by_year = cmp.groupby("year").apply(lambda d: pd.Series({
    "provinces": len(d),
    "ours_avg": d["fat_dynasty_share"].mean(),
    "ateneo_avg": d["ateneo_fat_share"].mean(),
    "correlation": d["fat_dynasty_share"].corr(d["ateneo_fat_share"])})).round(3)
print(f"province-years compared: {len(cmp):,} | overall correlation: {cmp['fat_dynasty_share'].corr(cmp['ateneo_fat_share']):.3f}")
by_year

province-years compared: 866 | overall correlation: 0.676


,provinces,ours_avg,ateneo_avg,correlation
year,,,,
1992,74.0,0.099,0.197,0.650
1995,76.0,0.116,0.219,0.711
1998,78.0,0.127,0.211,0.669
2001,79.0,0.145,0.218,0.598
2004,79.0,0.168,0.222,0.678
2007,80.0,0.194,0.223,0.553
2010,78.0,0.222,0.249,0.690
2013,80.0,0.232,0.263,0.734
2016,80.0,0.231,0.269,0.727


## Curated partitioning (one folder per election year)

In [7]:
import shutil
from pathlib import Path

C = Path("../data/curated")
officials = t[["person_uid", "year", "position", "province_std", "town_std", "party", "source",
               "is_dynastic", "is_fat_dynasty", "first_relative_year", "clan_id"]].copy()

tables = {
    "town_dynasty_stronghold": town,
    "province_dynasty_stronghold": prov_tbl,
    "official_terms_dynastic": officials,
}
for name, df in tables.items():
    df = df.copy()
    df["year"] = df["year"].astype(int)                  # plain int so the year folders read back cleanly
    for col in df.select_dtypes("category").columns:     # e.g. stronghold_level
        df[col] = df[col].astype(str)
    out = C / name
    if out.exists():
        shutil.rmtree(out)                               # rewrite cleanly each run (idempotent)
    df.to_parquet(out, partition_cols=["year"], index=False)
    parts = sorted(p.name for p in out.iterdir())
    print(f"{name:30s} {len(df):>8,} rows -> data/curated/{name}/  ({len(parts)} folders: {parts[0]} ... {parts[-1]})")

# Check: read back ONLY 2025 (this reads just one folder)
back = pd.read_parquet(C / "town_dynasty_stronghold", filters=[("year", "=", 2025)])
print(f"\nread back year=2025 only: {len(back):,} towns")

town_dynasty_stronghold          20,764 rows -> data/curated/town_dynasty_stronghold/  (13 folders: year=1988 ... year=2025)


province_dynasty_stronghold       1,021 rows -> data/curated/province_dynasty_stronghold/  (13 folders: year=1988 ... year=2025)


official_terms_dynastic         225,314 rows -> data/curated/official_terms_dynastic/  (13 folders: year=1988 ... year=2025)



read back year=2025 only: 1,637 towns
